# Jobs Finder

In [1]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
azure_open_api_key = os.getenv('AZURE_API_KEY')

if not azure_open_api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
import time

def scraper(url):
    # Optional headless
    options = Options()
    options.add_argument("--start-maximized")
    # options.add_argument("--headless=new")

    driver = webdriver.Chrome(options=options)

    driver.get(url)
    
    # sleep for 5 seconds to allow the page to load
    # time.sleep(5)

    # Now page is rendered
    content = driver.page_source
    driver.quit()
    
    return content
    

In [31]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from openai import OpenAI
import json

def job_finder(content,endpoint, deployment_name, api_key):
    
    client = OpenAI(
        base_url=endpoint,
        api_key=api_key
    )  
    completion = client.chat.completions.create(
        model=deployment_name,
        messages=[
            {"role": "system", 
            "content": """you are a helpful assistant that finds jobs(Internship, Trainee, Junior) that related to Devops, SRE, Cloud Engineering and give data in json format
                        here is the format you should give the data in
                        {
                            "jobs": [
                                {
                                    "Job Title": "SIntern Devops Engineer",
                                    "Mode": "hybrid or remote or on-site",                    
                                },
                                {
                                    "Job Title": "Site Reliability Engineer Intern",
                                    "Mode": "hybrid or remote or on-site or not_specified",                    
                                }
                            ]
                        }
                        
                        if there are no jobs in the content return  {}
                        Do not add normal positions like Devops Engineer or Site Reliability Engineer, only return internships.
                        if the mode of the job is not specified in the content, please write not_specified as the mode of the job
            """},
            {"role": "user", "content": f"""
                        here is the content of the jobs page {content}
                        please find the job title and mode of the jobsinds jobs(Internship, Trainee, Junior) that related to Devops, SRE, Cloud Engineering              and give the data in json format"""}
        ],
        response_format={"type": "json_object"}
    )
    
    # print(completion.choices[0].message.content)
    result=completion.choices[0].message.content
    return json.loads(result)

    

In [34]:
def jobs_display(urls, endpoint, deployment_name, api_key):
    my_jobs=[]
    for url in urls:
        print(f"Jobs from: {url['name']}")
        content=scraper(url['url'])
        jobs=job_finder(content=content, endpoint=endpoint, deployment_name=deployment_name, api_key=api_key)
        print(jobs)
        if 'jobs' not in jobs:
            continue
        for job in jobs['jobs']:
            my_jobs.append({
                "company": url['name'],
                "job_title": job['Job Title'],
                "mode": job['Mode']
            })
    return my_jobs
    

In [42]:
from company_urls import urls

endpoint = "https://my-openai-tkg.openai.azure.com/openai/v1"
deployment_name = "gpt-5.4-mini"
api_key = azure_open_api_key

urls = urls[:5]  
my_jobs=jobs_display(urls,endpoint=endpoint, deployment_name=deployment_name, api_key=api_key)

Jobs from: ws02
{'jobs': [{'Job Title': 'Intern - DevOps/Site Reliability Engineer (SRE)', 'Mode': 'not_specified'}]}
Jobs from: gtn
{'jobs': [{'Job Title': 'Intern - HR', 'Mode': 'not_specified'}]}
Jobs from: creativesoftware
{'jobs': []}
Jobs from: codegen
{}
Jobs from: gapstars
{'jobs': []}


In [43]:
import openai
from IPython.display import display, Markdown

endpoint = "https://my-openai-tkg.openai.azure.com/openai/v1"
deployment_name = "gpt-5.4-mini"
api_key = azure_open_api_key

client = OpenAI(
        base_url=endpoint,
        api_key=api_key
    )  

completion = client.chat.completions.create(
    model=deployment_name,
    messages=[
        {"role": "system", 
         "content": "you are getting a jobs array and you must give me markdown to display properly and beautifully"},
        {"role": "user", "content": f"""
                    here is the jobs array {my_jobs}"""}
    ],
   
)

output = completion.choices[0].message.content
display(Markdown(output))



```markdown
# Job Listings

## 1. ws02
- **Role:** Intern - DevOps/Site Reliability Engineer (SRE)
- **Mode:** Not specified

---

## 2. gtn
- **Role:** Intern - HR
- **Mode:** Not specified
```